# 📊 Live Processing Gallery
As the Ray cluster crunches through the 2,400+ images in the background, you can use this notebook to **visually track its progress in real-time**.

Run the cells below to generate an interactive gallery of the feathers being successfully extracted and saved to your `data/processed/` folder.

In [ ]:
import os
import glob
from collections import defaultdict
import ipywidgets as widgets
from IPython.display import display, HTML

RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

def refresh_status():
    raw_files = glob.glob(f"{RAW_DIR}/*.[jJ][pP][gG]")
    proc_files = glob.glob(f"{PROCESSED_DIR}/*_Feather_*.jpg")
    
    # Map extracted feathers back to their original image basename
    processed_map = defaultdict(list)
    for p in proc_files:
        base_name = os.path.basename(p).split("_Feather_")[0]
        processed_map[base_name].append(p)
        
    for k in processed_map:
        processed_map[k].sort() # Sort Feather_1, Feather_2...
        
    return raw_files, processed_map

raw_files, processed_map = refresh_status()

print("=========================================")
print(f"🎯 PROCESSING STATUS")
print("=========================================")
print(f"Total Raw Images in Queue: {len(raw_files)}")
print(f"Images Fully Processed:    {len(processed_map)} / {len(raw_files)}")
print(f"Individual Feathers Saved: {sum(len(v) for v in processed_map.values())}")
print("=========================================")

In [ ]:
# Re-run this cell anytime to fetch the latest images from the cluster!
raw_files, processed_map = refresh_status()
sorted_keys = sorted(list(processed_map.keys()))

def view_results(image_index=0):
    if not sorted_keys:
        display(HTML("<h3>Waiting for the first image to finish processing...</h3>"))
        return
        
    idx = min(image_index, len(sorted_keys) - 1)
    base_name = sorted_keys[idx]
    feathers = processed_map[base_name]
    
    html = f"<div style='background-color: #f8f9fa; padding: 15px; border-radius: 8px; margin-bottom: 20px;'>"
    html += f"<h2>Original Source: {base_name}</h2>"
    html += f"<p>Successfully extracted <b>{len(feathers)}</b> perfectly segmented individual feathers.</p>"
    html += "</div>"
    
    html += "<div style='display: flex; flex-wrap: wrap; gap: 15px; align-items: flex-end;'>"
    for i, f in enumerate(feathers):
        rel_url = f.replace("../", "")
        f_name = os.path.basename(f)
        
        # Render the LAST feather at 100% Native Scale
        img_style = "max-height: 350px; object-fit: contain;"
        if i == len(feathers) - 1:
            img_style = "max-height: none; max-width: none;"
            
        html += f"<div style='text-align: center; background: white; padding: 10px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);'>"
        html += f"<img src='../{rel_url}' style='{img_style} display: block; margin: 0 auto;' />"
        html += f"<p style='margin-top: 8px; font-family: monospace; font-weight: bold;'>{f_name}</p>"
        if i == len(feathers) - 1:
            html += f"<p style='color: red; font-size: 12px; font-weight: bold;'>🔍 100% NATIVE SCALE</p>"
        html += "</div>"
    
    html += "</div>"
    display(HTML(html))

if sorted_keys:
    widgets.interact(view_results, image_index=widgets.IntSlider(
        min=0, 
        max=len(sorted_keys)-1, 
        step=1, 
        value=0, 
        description='Browse:',
        layout=widgets.Layout(width='80%')
    ))
else:
    print("No processed images available yet. Please wait a moment and re-run this cell.")